# TensorFlow: Simple quadratic model (Keras)

* Using Keras

In [ ]:
import os
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from matplotlib import pyplot as plt
import matplotlib.ticker as mticker

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds

print("TF Version: ", tf.__version__)
print("TF Eager mode: ", tf.executing_eagerly())
print("TF GPU is", "available" if tf.config.list_physical_devices("GPU") else "not available")

## Generate training data

In [ ]:
(tr_ds, ts_ds), ds_info = tfds.load(
    "fashion_mnist",
    split=["train", "test"],
    with_info=True,
    as_supervised=True)

In [ ]:
BATCH_SIZE = 32

tr_ds = (tr_ds
    .cache()
    .shuffle(buffer_size=1000)
    .batch(BATCH_SIZE)
    .prefetch(buffer_size=tf.data.AUTOTUNE))

ts_ds = (ts_ds
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(buffer_size=tf.data.AUTOTUNE))

## Create a model

In [ ]:
def base_model():
    inputs = tf.keras.Input(shape=(28, 28, 1), name="digits")
    x = tf.keras.layers.Flatten()(inputs)
    x = tf.keras.layers.Rescaling(scale= 1.0 / 255)(x)
    x = tf.keras.layers.Dense(64, activation="relu", name="dense_1")(x)
    x = tf.keras.layers.Dense(64, activation="relu", name="dense_2")(x)
    outputs = tf.keras.layers.Dense(10, activation="softmax", name="predictions")(x)
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    return model

## Fit a model

In [ ]:
loss_object = tf.keras.losses.SparseCategoricalCrossentropy()
optimizer = tf.keras.optimizers.Adam()

In [ ]:
tr_acc_metric = tf.keras.metrics.SparseCategoricalAccuracy()
vl_acc_metric = tf.keras.metrics.SparseCategoricalAccuracy()

In [ ]:
# 1. Repeat for each training batch
#   - Calculate logits, loss on training set
#   - Calculate gradients of the loss w.r.t to model trainable weights
#   - Apply gradients on model using optimizer
#   - Accumulate accuracy metric
# 2. Repeat for each test batch
#   - Calculate logits, loss on test set
#   - Accumulate accuracy metric
# 3. Reset States of metrics
# 4. Calculate training and test loss for each epoch
# 5. Display epoch statistics

In [ ]:
def apply_gradient(optimizer, model, xs, ys):
    with tf.GradientTape() as tape:
        # Calculate logits, loss on training set
        logits = model(xs)
        loss_value = loss_object(y_true=ys, y_pred=logits)
    # Calculate gradients of the loss
    gradients = tape.gradient(loss_value, model.trainable_variables)
    # Apply gradients on model using optimizer
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return logits, loss_value

In [ ]:
def perform_validation(model, metric):
    losses = []
    for xs, ys in ts_ds:
        logits = model(xs)
        loss_value = loss_object(y_true=ys, y_pred=logits)
        losses.append(loss_value)
        # Accumulate accuracy metric
        metric.update_state(ys, logits)
    return losses

In [ ]:
def train_data_for_one_epoch(model, metric):
    losses = []
    for step, (xs, ys) in enumerate(tr_ds):
        # Calculate logits, loss on test set
        logits, loss_value = apply_gradient(optimizer, model, xs, ys)
        losses.append(loss_value)
        # Accumulate accuracy metric
        metric.update_state(ys, logits)
    return losses

In [ ]:
epochs = 10
model = base_model()
epochs_tr_losses, epochs_vl_losses = [], []
for epoch in range(epochs):
    print(f"Start of epoch: {epoch}")
    # Run through training data batches
    tr_losses = train_data_for_one_epoch(model, tr_acc_metric)
    tr_acc = tr_acc_metric.result()
    # Run through test data batches
    vl_losses = perform_validation(model, vl_acc_metric)
    vl_acc = vl_acc_metric.result()
    # Calculate losses mean
    tf_loss_mean = np.mean(tr_losses)
    epochs_tr_losses.append(tf_loss_mean)
    vl_loss_mean = np.mean(vl_losses)
    epochs_vl_losses.append(np.mean(vl_losses))
    # Reset states
    tr_acc_metric.reset_state()
    vl_acc_metric.reset_state()
    print(f"Epoch {epoch:02}:")
    print(f"...Train Loss={tf_loss_mean:.4f}, Val Loss={vl_loss_mean:.4f}")
    print(f"...Train Accuracy={tr_acc:.4f}, Val Accuracy={vl_acc:.4f}")
    print()

## Evaluate a model

### Evaluate loss

In [ ]:
def plot_metrics(train_metric, val_metric, metric_name, title, ylim=5):
  plt.title(title)
  plt.ylim(0, ylim)
  plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1))
  plt.plot(train_metric,color="blue",label=metric_name)
  plt.plot(val_metric,color="green",label="val_" + metric_name)

plot_metrics(epochs_tr_losses, epochs_vl_losses, "Loss", "Loss", ylim=1)

### Display confusion matrix

In [ ]:
# Split test data
it = list(ts_ds.unbatch().as_numpy_iterator())
test_xs = np.array([item[0] for item in it])
test_ys = np.array([item[1] for item in it])

In [ ]:
# Predict targets on test dataset
ys_pred = np.argmax(model(test_xs), axis=1)

In [ ]:
# Calculate confusion matrix
cm = confusion_matrix(
    y_true=test_ys,
    y_pred=ys_pred)

# Create confusion matrix display
ds = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=ds_info.features["label"].names)

In [ ]:
# Display confusion matrix
fig, ax = plt.subplots(figsize=(13, 13))
ds.plot(colorbar=False, ax=ax)
plt.show()